# 06 — Reporting

**Stage 7 of the Tier A pipeline.** Assembles `reports/final_report.html`: a single self-contained
file with every chart base64-embedded and every number read from `reports/_key_figures.json`, so
**nothing in the report is hardcoded**.

Structure follows `DOCS/STRUCTURE.md` §7: SCR storyline → Governing Thought → four MECE Key Lines →
a specific recommendation → one part per Key Line with action titles, exhibits and a "So What" under
each → appendix. Design follows `DOCS/DESIGN.md`: McKinsey navy/blue palette, Georgia display serif,
single 760px reading column, white figure cards in both themes.

In [1]:
import base64, json, re, warnings
from datetime import date
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "archive" / "Mall_Customers.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
REPORTS, FIGS = ROOT / "reports", ROOT / "reports" / "figures"

K = json.loads((REPORTS / "_key_figures.json").read_text())
print(f"{len(K)} key figures loaded from reports/_key_figures.json")


def pct(x, d=0):
    return f"{x*100:.{d}f}%"


def n(x):
    return f"{int(round(x)):,}"


def ci(c, d=2):
    return f"[{c[0]:.{d}f}, {c[1]:.{d}f}]"


def embed(name):
    b = base64.b64encode((FIGS / name).read_bytes()).decode()
    return f"data:image/png;base64,{b}"


SEG = {k: {s: K[f"seg_{k}"][s] for s in K["seg_n"]} for k in
       ["n", "income", "score", "age_median", "age_min", "age_max", "female", "share_odd"]}
UEA, AS, YE, BC, MS = "Under-engaged Affluent", "Affluent Spenders", "Young Enthusiasts", "Budget Conscious", "Mainstream"
for f in sorted(FIGS.glob("*.png")):
    print(f"  {f.name:28s} {f.stat().st_size/1024:6.0f} KB")

126 key figures loaded from reports/_key_figures.json
  ex01_construction.png            68 KB
  ex02_distributions.png           53 KB
  ex03_income_spread.png           53 KB
  ex04_age_ceiling.png             59 KB
  ex05_gender.png                  37 KB
  ex06_k_selection.png             54 KB
  ex07_segments.png                87 KB
  ex08_target_age.png              67 KB
  ex09_power.png                   65 KB


## 7.1 The storyline, finalised against the evidence

The Stage 0 SCR (`01` §0.5) was a hypothesis. Three parts changed:

1. The file is **not** a clean snapshot of a member base. It was generated in pairs (`01` §1.5, `04` §4.1), so the
   segment sizes cannot be used.
2. Income does not leave spending unaffected. It sets the *spread* of spending, not its level (`04` §4.2).
3. The "biggest conversion upside" is not the whole Under-engaged Affluent segment, only its under-40 part,
   because nobody over 40 scores above 60 (`04` §4.3, `05` §5.7).

In [2]:
SITUATION = (
    f"The mall scores its loyalty members from 1 to 100 on spending and records their age, gender and "
    f"income. In this file of {n(K['n_rows_raw'])} members the average score is about 50 at every income "
    f"level, and women and men score within {abs(K['score_mean_female'] - K['score_mean_male']):.0f} "
    "points of each other.")

COMPLICATION = (
    "Marketing wants to know whom to target. The file cannot answer that directly: it records no "
    "campaign response and no purchases, and the formula behind the Spending Score is not documented. The "
    "usual answer from this dataset, five income-and-spending segments, is then quoted with segment sizes "
    "as if they were shares of a real member base.")

GOVERNING_THOUGHT = (
    "The members fall into five segments that are real, stable and reproducible with two income and two "
    "score thresholds. But the file was generated in pairs, so its segment sizes mean nothing, and no "
    f"member over 40 scores above 60. Marketing should protect the {n(SEG['n'][AS])} Affluent Spenders, test "
    f"conversion first on the under-40 Under-engaged Affluent with a pilot run on the full member base, "
    "and size every budget from the real counts the rule produces.")

KEY_LINES = [
    ("The file",
     f"It was generated two members at a time. All {K['n_pairs_same_income']} consecutive ID pairs share an "
     f"income (random order matches at most {K['pairs_null_max']}), and at the income extremes the even-ID twin "
     f"outspends the odd-ID twin in {K['n_extreme_even_high']} of {K['n_extreme_pairs']} pairs. Each corner "
     "segment is made of one parity only, so <strong>the segment sizes come from the generator, not "
     "from a mall</strong>. The score is spread evenly from 1 to 99: it behaves like a rank, not like "
     "an amount spent."),
    ("The structure",
     f"Five segments, and much stronger than chance: silhouette {K['silhouette_k5']:.3f} against at most "
     f"{K['null_max_k5']:.2f} from data with no segments. A Gaussian mixture and Ward clustering find the same "
     f"groups (ARI {K['ari_gmm']:.2f} and {K['ari_ward']:.2f}), and every segment survives resampling "
     f"(bootstrap Jaccard &ge; {K['jaccard_min']:.2f}). Income 40k / 70k and score 40 / 60 reproduce the "
     f"clustering for <strong>{pct(K['rule_agreement'],1)}</strong> of members."),
    ("The drivers",
     f"Income does not move the average score (r = {K['r_income_score']:.2f} {ci(K['r_income_score_ci'])}), "
     f"but scores vary {K['sd_ratio_extremes']:.1f}&times; as much at the income extremes as in the middle. "
     f"Age is a ceiling: <strong>0 of {K['n_over_40']} members over 40 score above 60</strong>, where "
     f"{K['ceiling_expected']:.0f} would if age did not matter. Gender makes no measurable difference "
     f"(d = {K['gender_d']:.2f} {ci(K['gender_d_ci'])})."),
    ("The target",
     f"All {SEG['n'][AS]} Affluent Spenders are aged {SEG['age_min'][AS]}&ndash;{SEG['age_max'][AS]} with a mean "
     f"score of {SEG['score'][AS]:.0f}: they are the core to retain. Of the {SEG['n'][UEA]} Under-engaged "
     f"Affluent (mean score {SEG['score'][UEA]:.0f}), only <strong>{K['n_uea_young']} are 40 or under</strong>, "
     "where high-spending members of the same age and income exist. Proving a lift needs about "
     f"{n(K['n_per_arm_5_50'])} members per arm, so the pilot must run on the full member base."),
]

RECOMMENDATION = [
    ("Run the two-threshold rule on the full loyalty base and publish real segment counts.",
     "One query (income &ge; 70k or &le; 40k; score &ge; 60 or &le; 40) assigns every member. Use those "
     f"counts, never this file's {pct(SEG['n'][UEA]/K['n_rows_raw'])} / {pct(SEG['n'][AS]/K['n_rows_raw'])} "
     "shares, for any budget. <em>Owner: CRM analytics lead, within two weeks.</em>"),
    ("Protect the Affluent Spenders without discounting them.",
     "They already spend. Offer recognition instead of price: early access, a loyalty tier, events. Track "
     "their score month by month as the retention measure. <em>Owner: CRM manager, from next quarter.</em>"),
    ("Pilot conversion on the under-40 Under-engaged Affluent.",
     "Randomise members to offer or holdout within the segment, stratified by gender. The primary metric is "
     "redemption or incremental spend from transactions, written down before launch. Size each arm from "
     f"the power table (about {n(K['n_per_arm_5_50'])} per arm for a 50% lift on a 5% response) and run 8&ndash;12 "
     "weeks with no early stopping. Test the over-40s in a separate, cheaper arm. <em>Owner: Head of CRM "
     "&amp; Marketing, design approved this month.</em>"),
    ("Keep the other three segments on low-cost plans.",
     f"Mainstream is the largest and most uniform group (score SD {K['band_sd']['41–69k']:.1f} in the middle "
     "income band), so one broad message fits it. Grow Young Enthusiasts with affordability guardrails: no "
     "credit or instalment push to members on &le; 40k. Take Budget Conscious out of paid media. "
     "<em>Owner: campaign managers, next campaign calendar.</em>"),
    ("Get the score definition and transaction data before the pilot reads out.",
     "Without them, &ldquo;spending&rdquo; stays a number the mall assigns. Legal should confirm that "
     "age-based campaign selection is allowed under the mall's privacy policy. <em>Owner: loyalty platform "
     "team and legal, before pilot launch.</em>"),
]
print("Governing Thought:\n ", re.sub(r"&\w+;", "-", GOVERNING_THOUGHT))
print(f"\n{len(KEY_LINES)} key lines · {len(RECOMMENDATION)} recommendations")

Governing Thought:
  The members fall into five segments that are real, stable and reproducible with two income and two score thresholds. But the file was generated in pairs, so its segment sizes mean nothing, and no member over 40 scores above 60. Marketing should protect the 39 Affluent Spenders, test conversion first on the under-40 Under-engaged Affluent with a pilot run on the full member base, and size every budget from the real counts the rule produces.

4 key lines · 5 recommendations


### MECE check on the Key Lines

| Key Line | Object | Overlap? |
|---|---|---|
| 1 — The file | the instrument: how it was built, what the score is | no claim about members |
| 2 — The structure | whether segments exist and are stable | grouping only, no attributes, no action |
| 3 — The drivers | which member attributes relate to spending | attributes only, no grouping |
| 4 — The target | whom to act on and how to prove it | action only |

Together they cover branches A–D of the Stage 0 issue tree (`01` §0.4).

In [3]:
EXHIBITS = [
    dict(part=1, file="ex01_construction.png",
         title="Every income is shared by two members; at both extremes one twin spends and the other does not",
         sowhat=f"The {K['n_rows_raw']} rows are {K['n_pairs']} pairs with identical income, sorted by income before IDs "
                f"were given out. At income &ge; 70k the odd-ID twin scores at most {K['hi_income_odd_max']} and the "
                f"even-ID twin at least {K['hi_income_even_min']}, without exception. The four corner segments were "
                f"filled one twin at a time, which is why Affluent Spenders ({SEG['n'][AS]}) and Under-engaged "
                f"Affluent ({SEG['n'][UEA]}) are nearly the same size, as are Young Enthusiasts ({SEG['n'][YE]}) and "
                f"Budget Conscious ({SEG['n'][BC]}). <strong>Use this file for the shape of the segments, never "
                "for their size.</strong>"),
    dict(part=2, file="ex06_k_selection.png",
         title=f"Five segments, and much stronger than chance: silhouette {K['silhouette_k5']:.3f} against at most "
               f"{K['null_max_k5']:.2f} from data with no segments",
         sowhat=f"Silhouette, Davies&ndash;Bouldin and the mixture-model BIC all choose five. Calinski&ndash;Harabasz makes "
                f"its largest gain at five (+{K['ch_gain_k5']:.0f}) and creeps up to a peak at eight. The grey band is "
                "what k-means finds in 200 data sets with no segments (evenly spread points, or real values shuffled "
                "against each other). At k = 5 the real data clears the highest of them by more than 0.1. Adding age "
                f"as a third input makes every k worse (best {K['silhouette_3d_best']:.2f}), so age describes the "
                "segments but does not define them."),
    dict(part=3, file="ex03_income_spread.png",
         title="Income does not set how much members spend; it sets how much their spending varies",
         sowhat=f"Every income band averages a score of about 50 (r = {K['r_income_score']:.2f}). But in the 41&ndash;69k "
                f"band the score's standard deviation is {K['band_sd']['41–69k']:.1f} points, against "
                f"{K['band_sd']['≤ 40k']:.1f} at &le; 40k and {K['band_sd']['≥ 70k']:.1f} at &ge; 70k, a "
                f"{K['sd_ratio_extremes']:.1f}&times; ratio {ci(K['sd_ratio_ci'],1)}. Distance correlation "
                f"({K['dcor_income_score']:.2f}, permutation p = {K['dcor_p']:.3f}) confirms the two are strongly "
                "dependent even though the linear correlation is zero. <strong>Income tells marketing where "
                "targeting matters:</strong> in the middle band one message fits almost everyone; at either end "
                "the same income holds opposite behaviours, so the offer has to be chosen by score."),
    dict(part=3, file="ex04_age_ceiling.png",
         title=f"No member over 40 scores above 60: high spending is an under-40 behaviour in this file",
         sowhat=f"{K['n_under_40_hi']} of {K['n_under_40']} members aged 40 or under score above 60. Of the "
                f"{K['n_over_40']} over 40, none does; age-blind chance would put {K['ceiling_expected']:.0f} there "
                f"(Fisher p = {K['ceiling_p']:.0e}; the 95% upper bound on the over-40 rate is "
                f"{pct(K['ceiling_rate_ci'][1],1)}). The ceiling holds in every income band, and age is unrelated "
                f"to income (r = {K['r_age_income']:.2f}), so it is not income in disguise. The overall correlation "
                f"(&rho; = {K['rho_age_score']:.2f}) understates it: this is a wall, not a slope."),
    dict(part=4, file="ex08_target_age.png",
         title=f"Every Affluent Spender is {SEG['age_min'][AS]}&ndash;{SEG['age_max'][AS]}; only {K['n_uea_young']} of the "
               f"{SEG['n'][UEA]} Under-engaged Affluent are in that age range",
         sowhat=f"The under-40 Under-engaged Affluent earn the same as the Affluent Spenders (mean "
                f"{K['uea_young_income']:.0f}k vs {SEG['income'][AS]:.0f}k) and are the same age (median "
                f"{K['uea_young_age_median']:.0f} vs {SEG['age_median'][AS]:.0f}). Their high-spending counterpart "
                f"exists in the data. The {K['n_uea_old']} over 40 (median age {K['uea_old_age_median']:.0f}) have no "
                "high-spending counterpart anywhere in the file. They are not written off: they get a separate, "
                "cheaper test, with lower expectations."),
    dict(part=4, file="ex09_power.png",
         title=f"A credible pilot needs the full member base: a 50% lift on a 5% response takes about "
               f"{n(K['n_per_arm_5_50'])} members per arm",
         sowhat=f"The {K['pilot_slice_n']} members in this file's pilot slice could only detect an offer that lifts a 5% "
                f"response rate to {pct(K['pilot_mde_from_5'])}. Real campaigns move response by a fraction of "
                f"that. At a 10% baseline a 50% lift needs {n(K['power_table']['10% baseline']['+50% lift'])} per arm; "
                f"at 2% it needs {n(K['power_table']['2% baseline']['+50% lift'])}. If the real segment is smaller "
                "than two arms' worth, the pilot runs longer, accepts a larger minimum lift, or pools both affluent "
                "age groups with age as a stratification variable."),
]
print(f"{len(EXHIBITS)} exhibits, all files present: {all((FIGS / e['file']).exists() for e in EXHIBITS)}")

6 exhibits, all files present: True


## 7.2 Design tokens — `DOCS/DESIGN.md`, consumed not re-picked

The four segment colours were **validated with the dataviz skill's checker**, not eyeballed:
`node scripts/validate_palette.js "#2251FF,#008A7E,#C1841C,#00A9F4" --mode light --pairs all` passes
lightness, chroma floor, CVD separation (worst pair ΔE 12.5 protan) and normal-vision separation (18.3).
Cyan's contrast on white is 2.56:1 (WARN), so every segment is also direct-labelled and has its own
marker shape, and the profile is given as a table. DESIGN.md's Teal `#00857C` **fails the chroma floor**
(C = 0.097 < 0.10), so it is nudged to `#008A7E` on the same hue.

In [4]:
CSS = """
:root{
  --navy:#051C2C; --accent:#2251FF; --accent-soft:#eef2ff; --teal:#008A7E; --amber:#C1841C;
  --gold:#8A5E12; --slate:#7F93A6; --grey:#9FADB8;
  --ground:#f4f6f8; --surface:#ffffff; --ink:#051C2C; --ink-2:#43596b; --rule:#dfe5ea;
}
@media (prefers-color-scheme: dark){
  :root:not([data-theme="light"]){ --ground:#051C2C; --surface:#0c2434; --ink:#e9eef2; --ink-2:#a9bccb;
         --accent:#5a8dff; --accent-soft:#12304a; --rule:#1d3a4e; --gold:#E5B84A; }
}
:root[data-theme="light"]{
  --ground:#f4f6f8; --surface:#ffffff; --ink:#051C2C; --ink-2:#43596b;
  --accent:#2251FF; --accent-soft:#eef2ff; --rule:#dfe5ea; --gold:#8A5E12;
}
:root[data-theme="dark"]{
  --ground:#051C2C; --surface:#0c2434; --ink:#e9eef2; --ink-2:#a9bccb;
  --accent:#5a8dff; --accent-soft:#12304a; --rule:#1d3a4e; --gold:#E5B84A;
}
*{box-sizing:border-box}
body{margin:0;background:var(--ground);color:var(--ink);
  font-family:'Helvetica Neue',Arial,system-ui,'Segoe UI',sans-serif;
  font-size:16px;line-height:1.62;-webkit-font-smoothing:antialiased}
.wrap{max-width:760px;margin:0 auto;padding:56px 22px 96px}
h1{font-family:Georgia,'Times New Roman',serif;font-weight:400;font-size:2.35rem;line-height:1.18;
  margin:0 0 .5rem;text-wrap:balance;letter-spacing:-.01em}
h2{font-size:1.32rem;margin:3.2rem 0 .6rem;text-wrap:balance;line-height:1.3}
h3{font-size:1.02rem;margin:2rem 0 .5rem;text-wrap:balance}
p{margin:0 0 1rem;max-width:68ch}
a{color:var(--accent)}
.eyebrow{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.7rem;
  letter-spacing:.16em;text-transform:uppercase;color:var(--slate);margin:0 0 .6rem}
.sub{color:var(--ink-2);font-size:1.02rem;margin-bottom:2.2rem}
.answer{background:var(--surface);border-left:4px solid var(--accent);border-radius:0 6px 6px 0;
  padding:26px 28px;margin:2rem 0 2.6rem;box-shadow:0 1px 3px rgba(5,28,44,.07)}
.scr{margin:0 0 1.3rem}
.scr b{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.68rem;letter-spacing:.14em;
  text-transform:uppercase;color:var(--slate);display:block;margin-bottom:.2rem}
.gt{font-family:Georgia,'Times New Roman',serif;font-size:1.22rem;line-height:1.5;
  color:var(--accent);border-top:1px solid var(--rule);border-bottom:1px solid var(--rule);
  padding:1.1rem 0;margin:1.4rem 0}
ol.keylines{padding-left:0;counter-reset:kl;list-style:none;margin:1.4rem 0 0}
ol.keylines li{counter-increment:kl;position:relative;padding-left:2.4rem;margin-bottom:1.05rem}
ol.keylines li::before{content:counter(kl);position:absolute;left:0;top:.05rem;
  font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.78rem;font-weight:700;
  color:var(--accent);border:1.5px solid var(--accent);border-radius:50%;
  width:1.5rem;height:1.5rem;display:grid;place-items:center}
ol.keylines b{display:block;font-size:.72rem;letter-spacing:.13em;text-transform:uppercase;
  color:var(--slate);font-family:ui-monospace,'SF Mono',Consolas,monospace;margin-bottom:.15rem}
.tiles{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin:2.2rem 0}
.tile{background:var(--surface);border:1px solid var(--rule);border-radius:6px;padding:14px 14px 12px}
.tile .v{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:1.28rem;font-weight:700;
  color:var(--accent);font-variant-numeric:tabular-nums;line-height:1.15}
.tile .l{font-size:.74rem;color:var(--ink-2);margin-top:.3rem;line-height:1.35}
.rec{background:var(--surface);border:1px solid var(--gold);border-left:4px solid var(--gold);
  border-radius:0 6px 6px 0;padding:20px 24px;margin:2rem 0}
.rec .eyebrow{color:var(--gold)}
.rec ol{padding-left:1.15rem;margin:.6rem 0 0}
.rec li{margin-bottom:.75rem}
.rec li b{color:var(--ink)}
.part{border-top:2px solid var(--accent);margin:4.2rem 0 0;padding-top:.7rem}
.figcard{background:#fff;border:1px solid var(--rule);border-radius:6px;padding:16px;margin:1.4rem 0 0;
  overflow-x:auto}
.figcard img{width:100%;max-width:100%;display:block}
.sowhat{border-left:3px solid var(--accent);background:var(--accent-soft);border-radius:0 5px 5px 0;
  padding:13px 17px;margin:.85rem 0 2.4rem;font-size:.93rem;color:var(--ink)}
.sowhat b.k{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.68rem;letter-spacing:.13em;
  text-transform:uppercase;color:var(--accent)}
.tblwrap{overflow-x:auto;margin:1.2rem 0}
table{border-collapse:collapse;width:100%;font-size:.88rem}
th{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.68rem;letter-spacing:.1em;
  text-transform:uppercase;color:var(--slate);text-align:left;border-bottom:1.5px solid var(--rule);
  padding:8px 10px;font-weight:600}
td{border-bottom:1px solid var(--rule);padding:8px 10px;font-variant-numeric:tabular-nums;
  vertical-align:top}
tr.best td{background:var(--accent-soft)}
code{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.87em;
  background:var(--accent-soft);padding:.1em .32em;border-radius:3px}
.foot{margin-top:4.5rem;padding-top:1.3rem;border-top:1px solid var(--rule);
  font-size:.8rem;color:var(--slate)}
@media (max-width:620px){ .tiles{grid-template-columns:repeat(2,1fr)} h1{font-size:1.85rem}
  .wrap{padding:36px 16px 72px} .answer{padding:20px 18px} }
"""
print(f"CSS: {len(CSS):,} chars")

CSS: 5,167 chars


## 7.3 Assemble

In [5]:
def table(df, best_row=None):
    h = "".join(f"<th>{c}</th>" for c in df.columns)
    body = ""
    for i, (_, r) in enumerate(df.iterrows()):
        cls = ' class="best"' if best_row is not None and i == best_row else ""
        body += f"<tr{cls}>" + "".join(f"<td>{v}</td>" for v in r) + "</tr>"
    return f'<div class="tblwrap"><table><thead><tr>{h}</tr></thead><tbody>{body}</tbody></table></div>'


def sowhat(txt):
    return f'<div class="sowhat"><b class="k">So what</b><br>{txt}</div>'


TILES = [
    (f"{K['silhouette_k5']:.3f}", f"segment separation (silhouette), against at most {K['null_max_k5']:.2f} in data with no segments"),
    (pct(K["rule_agreement"], 1), "of members placed in the same segment by two income and two score thresholds"),
    (f"0 / {K['n_over_40']}", f"members over 40 who score above 60 &mdash; {K['ceiling_expected']:.0f} expected by chance"),
    (f"{K['n_uea_young']} of {SEG['n'][UEA]}", "Under-engaged Affluent aged 40 or under &mdash; the first conversion pilot"),
]

file_tbl = pd.DataFrame({
    "What the data review says": [
        "<code>CustomerID</code>: &ldquo;unique, sequential, no information content&rdquo;",
        "No missing values, no duplicates; clean, &ldquo;synthetic-looking&rdquo;",
        "Cluster sizes 39 / 35 / 22 / 23 / 81; Under-engaged Affluent = &ldquo;biggest untapped potential&rdquo;",
        "Spending Score: mall-assigned from behaviour and purchase data",
    ],
    "What the file shows": [
        f"It is the income rank (Spearman {K['id_income_spearman']:.2f}), and its parity decides the corner segment "
        f"(odd IDs: {pct(SEG['share_odd'][UEA])} of Under-engaged Affluent, {pct(SEG['share_odd'][AS])} of Affluent Spenders)",
        f"Confirmed, and more specific: {K['n_pairs_same_income']} of {K['n_pairs']} ID pairs share an income, "
        f"against at most {K['pairs_null_max']} in 10,000 random orderings",
        "The pairs were split into opposite corners, so the four corner sizes are near-equal by construction "
        "and are not estimates of anything",
        f"Spread almost evenly over 1&ndash;99 (skew {K['score_skew']:.2f}). Real spending is right-skewed, so this "
        "behaves like a rank. Quote score points, never &ldquo;spends N&times; more&rdquo;",
    ],
})

robust_tbl = pd.DataFrame({
    "Check": ["Same settings, 50 random seeds", "Single random start (200 runs)", "Gaussian mixture, k = 5",
              "Ward hierarchical, k = 5", "200 bootstrap refits", "Two income outliers (137k) removed",
              "Naive median-split quadrants", "Members on a segment border (silhouette &lt; 0.25)"],
    "Result": [f"ARI {K['seed_ari_min']:.2f} (identical)", f"{pct(K['single_init_fail'])} land on another solution",
               f"ARI {K['ari_gmm']:.2f}", f"ARI {K['ari_ward']:.2f}",
               f"Jaccard {K['jaccard_min']:.2f}&ndash;{K['jaccard_max']:.2f} per segment",
               f"ARI {K['ari_wo_outliers']:.2f}", f"ARI {K['ari_quadrant']:.2f} &mdash; misses the structure",
               f"{K['n_borderline']} of {K['n_rows_raw']}"],
})

driver_tbl = pd.DataFrame({
    "Attribute": ["Income &rarr; average score", "Income &rarr; spread of score", "Age &rarr; score",
                  "Age over 40 &rarr; score above 60", "Gender &rarr; score"],
    "Effect [95% CI]": [f"r = {K['r_income_score']:.2f} {ci(K['r_income_score_ci'])}",
                        f"SD ratio {K['sd_ratio_extremes']:.1f}&times; {ci(K['sd_ratio_ci'],1)}",
                        f"&rho; = {K['rho_age_score']:.2f} {ci(K['rho_age_score_ci'])}",
                        f"0 of {K['n_over_40']}; rate {ci([x*100 for x in K['ceiling_rate_ci']],1)}%",
                        f"d = {K['gender_d']:.2f} {ci(K['gender_d_ci'])}"],
    "Corrected p": [f"{K['income_level_p_bh']:.2f}", "&lt; 0.001", "&lt; 0.001", "&lt; 0.001", f"{K['gender_p_bh']:.2f}"],
    "Reading": ["none", "strong", "moderate", "a hard ceiling", "none"],
})

order = [AS, UEA, YE, BC, MS]
action = {AS: "retain", UEA: f"pilot: {K['n_uea_young']} aged &le; 40 first", YE: "grow, with guardrails",
          BC: "deprioritise", MS: "one broad message"}
profile_tbl = pd.DataFrame({
    "Segment": order,
    "n (file only)": [SEG["n"][s] for s in order],
    "Income (k$)": [f"{SEG['income'][s]:.0f}" for s in order],
    "Score": [f"{SEG['score'][s]:.0f}" for s in order],
    "Ages": [f"{SEG['age_min'][s]}&ndash;{SEG['age_max'][s]}" for s in order],
    "Female": [pct(SEG["female"][s]) for s in order],
    "Action": [action[s] for s in order],
})

parts = {1: "Part I &middot; The file", 2: "Part II &middot; The structure",
         3: "Part III &middot; The drivers", 4: "Part IV &middot; The target"}
part_h2 = {
    1: ("This file can draw the segments but not size them: it was generated in pairs", file_tbl, None),
    2: ("The five segments are real, stable, and reducible to two thresholds on each axis", robust_tbl, None),
    3: ("Income decides how much spending varies, age caps it at 40, and gender does not matter", driver_tbl, None),
    4: (f"Retain the Affluent Spenders, pilot the under-40 Under-engaged Affluent, and size the pilot on the real base",
        profile_tbl, 1),
}

html = ['<div class="wrap">']
html.append('<p class="eyebrow">Loyalty marketing review &middot; Tier A</p>')
html.append("<h1>Keep the Affluent Spenders, test the young Under-engaged Affluent, and size nothing from this file</h1>")
html.append('<p class="sub">Mall customer segmentation &mdash; '
            f'{n(K["n_rows_raw"])} loyalty members, annual income and a mall-assigned Spending Score. '
            'Which segments marketing should retain or try to convert, and how far this file can be '
            'trusted to size them.</p>')

html.append('<div class="answer">')
for lbl, txt in [("Situation", SITUATION), ("Complication", COMPLICATION)]:
    html.append(f'<p class="scr"><b>{lbl}</b>{txt}</p>')
html.append(f'<p class="gt">{GOVERNING_THOUGHT}</p>')
html.append('<ol class="keylines">')
for title, body in KEY_LINES:
    html.append(f"<li><b>{title}</b>{body}</li>")
html.append("</ol></div>")

html.append('<div class="tiles">')
for v, l in TILES:
    html.append(f'<div class="tile"><div class="v">{v}</div><div class="l">{l}</div></div>')
html.append("</div>")

html.append('<div class="rec"><p class="eyebrow">Recommendation &mdash; Head of CRM &amp; Marketing</p><ol>')
for head, body in RECOMMENDATION:
    html.append(f"<li><b>{head}</b> {body}</li>")
html.append("</ol></div>")

html.append('<div class="figcard"><img alt="Exhibit 7 — the five segments and the two-threshold rule" src="'
            + embed("ex07_segments.png") + '"></div>')
html.append(sowhat(
    f"The key exhibit. K-means on income and score finds five segments; the dashed lines at income 40k / 70k "
    f"and score 40 / 60 put {pct(K['rule_agreement'],1)} of members in the same segment, and the "
    f"{K['n_rule_disagree']} exceptions all sit on a border. The CRM team can segment the full base with that "
    f"rule and no model. The counts printed here belong to this file only. Part I explains why they must "
    f"not be used to size anything."))

for p_no in (1, 2, 3, 4):
    html.append(f'<div class="part"><p class="eyebrow">{parts[p_no]}</p></div>')
    h2, tbl, best = part_h2[p_no]
    html.append(f"<h2>{h2}</h2>")
    if tbl is not None:
        html.append(table(tbl, best_row=best))
    for e in EXHIBITS:
        if e["part"] != p_no:
            continue
        html.append(f'<h3>{e["title"]}</h3>')
        html.append(f'<div class="figcard"><img alt="{re.sub("&[a-z]+;", "-", e["title"])}" src="{embed(e["file"])}"></div>')
        html.append(sowhat(e["sowhat"]))
    if p_no == 4:
        html.append(
            f"<p><strong>Fairness.</strong> Segments do not differ by gender (&chi;&sup2; p = {K['gender_seg_p']:.2f}, "
            f"Cram&eacute;r's V = {K['gender_seg_v']:.2f}), and the rule uses only income and score. The under-40 pilot "
            f"slice is {K['pilot_n_female']} of {K['pilot_slice_n']} female ({pct(K['pilot_female'])}) against "
            f"{pct(K['rest_female'])} in the rest of the file. With 15 members that is within chance (p = "
            f"{K['pilot_gender_p']:.2f}), but a real pilot should randomise within gender and report by gender. "
            "Age decides only the <em>order</em> of testing; over-40s are not excluded from offers.</p>")

html.append('<div class="part"><p class="eyebrow">Appendix &middot; Method and limits</p></div>')
html.append("<h2>Hypotheses that did not survive</h2>")
html.append(
    "<p><strong>&ldquo;The file is a clean snapshot of a member base.&rdquo;</strong> Rejected. It was generated in "
    "income-sorted pairs, and the data review's &ldquo;no information content&rdquo; for the ID is wrong. The "
    "segment shapes stand; the segment sizes do not.</p>")
html.append(
    "<p><strong>&ldquo;Income does not drive spending.&rdquo;</strong> Half right. It does not set the average "
    "score, but it sets the spread, which is what tells marketing where targeting matters.</p>")
html.append(
    "<p><strong>&ldquo;The Under-engaged Affluent are the biggest conversion upside.&rdquo;</strong> Narrowed. Only "
    f"the {K['n_uea_young']} aged 40 or under have high-spending peers of the same age and income in the file. "
    "&ldquo;Convertible&rdquo; is still a hypothesis until the pilot reads out.</p>")
html.append("<h2>What this analysis does not claim</h2>")
html.append(
    "<p><strong>No response, no causation.</strong> The file has no campaign exposure, response or "
    "transactions. Segments describe how members score now, not how they would react to an offer. The "
    "pilot design (Part IV) is how that question gets answered.</p>")
html.append(
    "<p><strong>A teaching dataset, not a mall.</strong> 200 members, generated in pairs, with an undefined "
    "score and income whose meaning (individual or household) is undocumented. The thresholds are in this "
    "file's units and should be re-checked if the real score is computed differently.</p>")
html.append(
    f"<p><strong>Cleaning.</strong> Nothing was imputed or dropped: the file has no nulls or duplicate members. "
    f"Two income values (137k) above the Tukey fence were flagged and kept; removing them changes nothing "
    f"(ARI {K['ari_wo_outliers']:.2f}). <code>CustomerID</code> and its parity were barred from every feature "
    f"set. The log is <code>data/processed/_cleaning_log.csv</code>.</p>")
html.append(
    f"<p><strong>Statistics.</strong> {K['n_tests']} hypothesis tests were corrected together "
    f"(Benjamini&ndash;Hochberg); {K['n_significant_bh']} survive. Effect sizes carry bootstrap (2,000 resamples), "
    "Fisher-z or Wilson confidence intervals. Non-linear dependence is tested by distance correlation with a "
    "permutation null. Because rows are paired by construction, p-values describe this file, not a "
    "population.</p>")
html.append(
    f"<p><strong>Segmentation.</strong> K-means (k = 5, <code>n_init = 10</code>) on standardised income and "
    f"score. k chosen by silhouette, Davies&ndash;Bouldin and mixture-model BIC, and tested against 200 "
    f"no-structure data sets. Stability: 50 seeds, 200 bootstrap refits, two other algorithms. A single random "
    f"start lands elsewhere {pct(K['single_init_fail'])} of the time, so <code>n_init</code> must stay &ge; 10. "
    f"Pilot sizes come from a two-proportion power calculation (two-sided &alpha; = 0.05, power 0.80). The "
    "segmentation card is <code>models/segmentation_card.json</code>.</p>")
html.append(
    "<p><strong>Reproducibility.</strong> Six numbered notebooks run in order with "
    "<code>RANDOM_STATE = 42</code> throughout, and a restart-and-run-all from the raw CSV is verified. "
    "They import no local modules: chart styling and every helper are defined inline. Every number in "
    "this report is read from <code>reports/_key_figures.json</code>, written by notebooks 01&ndash;05.</p>")

html.append(
    f'<p class="foot">Source: Kaggle &ldquo;Mall Customer Segmentation Data&rdquo; &mdash; '
    f'{n(K["n_rows_raw"])} rows, SHA-256 <code>{K["sha256"][:12]}&hellip;</code>. Analysis '
    f'{date.today().isoformat()}. Built to the McKinsey communication standard per '
    f'<code>DOCS/STRUCTURE.md</code>; visual system per <code>DOCS/DESIGN.md</code>.</p>')
html.append("</div>")

doc = ("<!doctype html><html lang=\"en\"><head><meta charset=\"utf-8\">"
       "<meta name=\"viewport\" content=\"width=device-width,initial-scale=1\">"
       "<title>Mall Customer Segmentation</title>"
       f"<style>{CSS}</style></head><body>{''.join(html)}</body></html>")

out = REPORTS / "final_report.html"
out.write_text(doc, encoding="utf-8")
print(f"wrote {out.relative_to(ROOT)}  ({len(doc)/1024/1024:.2f} MB, self-contained)")
print(f"  {len(EXHIBITS) + 1} exhibits base64-embedded · 0 external requests")

wrote reports\final_report.html  (0.61 MB, self-contained)
  7 exhibits base64-embedded · 0 external requests


In [6]:
assert 'src="data:image/png;base64,' in doc
assert doc.count("<img") == len(EXHIBITS) + 1
for bad in ["http://", "https://", "<script"]:
    assert bad not in doc, f"external or scripted content found: {bad}"
placeholders = re.findall(r"\{[a-zA-Z_\[\]']+\}", doc.split("<style>")[1].split("</style>")[1])
assert not placeholders, f"unformatted placeholders: {placeholders}"
assert "nan" not in re.sub(r"<[^>]+>", " ", doc.split("</style>")[1]).split(), "a NaN leaked into the text"
print("self-containment checks pass:")
print("  all images inline base64 · no external requests · no scripts · no unformatted placeholders · no NaN")
print(f"  {doc.count('<img')} images, {len(doc):,} chars")

self-containment checks pass:
  all images inline base64 · no external requests · no scripts · no unformatted placeholders · no NaN
  7 images, 641,969 chars


## Stage 7 — Gate Checklist (`DOCS/STRUCTURE.md`)

- [x] **SCR finalised** — evidence-backed; every figure read from `_key_figures.json`
- [x] **Governing Thought exists** — answers the business question in `01` §0.1 (segments, retain/convert, how far to trust sizes)
- [x] **Key Lines are MECE** — 4 arguments (file / structure / drivers / target), table above
- [x] **One-Page Test** — SCR, Governing Thought, 4 Key Lines, 4 stat tiles, a 5-point recommendation
      and the key exhibit all sit above the first part divider
- [x] **All titles are Action Titles** — every `h2`/`h3` states the insight
- [x] **Every exhibit has a "So What"** — 6 exhibits + the key exhibit, 7 annotations
- [x] **Vertical logic holds** — the four `h2`s alone tell the argument
- [x] **Horizontal logic holds** — each part's table and exhibits support its `h2`
- [x] **No orphan findings** — ex02 (distributions) and ex05 (gender) live in `03` and are summarised in
      Part I's table and Part III's driver table; the three hypotheses that failed are reported, not dropped
- [x] **Recommendations specific** — five, each with an owner and a timing
- [x] **Methodology in the appendix** · [x] **Reproducible** — pinned packages, seeds, relative paths, no local imports

## Tier A Quality Checklist

- [x] Problem statement explicit and linked to a decision (budget split across segments, pilot design)
- [x] Methodology appropriate — Path A tests, unsupervised segmentation, Path C only as a prospective design
- [x] Assumptions stated and tested (independence violated by construction; stated)
- [x] Uncertainty quantified — bootstrap / Fisher-z / Wilson CIs; no-structure nulls for the silhouette
- [x] Alternative explanations addressed — chance structure (§5.2), algorithm artefact (§5.4), outliers (§5.4),
      income confounding the age ceiling (§4.3), generator construction (§4.1)
- [x] Fairness: gender independence, pilot-slice skew, scoped age use · [x] Quantified language · [x] Dependencies pinned